#### Cleaning Section: [cleaning_report.md](../docs/Cleaning_logs/cleaning_report.md)

- `Early Data Analysis`
- `Duplicate Correction`
- `Inconsistencies Fixing`
- `Missing Data Checking`
- `Planning for EDA`

In [1]:
# Data was given in snippets, we have to concat everything into one df
import pandas as pd
import plotly.express as px

years = list(range(2020, 2027))
dfs = []

for year in years:
    for quarter in ["Q1", "Q2", "Q3", "Q4"]:
        file_path = f"../data/raw/BDD PRODUCCION/BDD PRODUCCION/{year}/{quarter} {year}.csv"
        try:
            df = pd.read_csv(file_path)

            if year == 2025 and quarter == "Q2":
                s = df["order_date"]
                p1 = pd.to_datetime(s, dayfirst=True, errors="coerce")
                p2 = pd.to_datetime(s, dayfirst=False, errors="coerce")
                num = pd.to_numeric(s, errors="coerce")
                excel = pd.to_datetime(num, unit="D", origin="1899-12-30")
                # Fill order_date with p2 first, then p1, and finally excel if both are NaT 
                df["order_date"] = p2.fillna(p1).fillna(excel)
            else:
                df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

            dfs.append(df)
        except FileNotFoundError as e:
            print(f"File not found: {file_path}, error: {e}")

remissions = pd.concat(dfs, ignore_index=True)
remissions.head()


File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q2 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q2 2026.csv'
File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q3 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q3 2026.csv'
File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q4 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q4 2026.csv'


,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,2020,51013232,2020-02-04,1073,2/4/2020 7:00,7044,510,6.0,2/4/2020 6:45,2/4/2020 8:17,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
1,2020,51013233,2020-02-04,1073,2/4/2020 7:00,6603,510,5.5,2/4/2020 6:45,2/4/2020 8:26,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
2,2020,51013235,2020-02-04,1028,2/4/2020 8:00,9631,510,2.5,2/4/2020 7:37,2/4/2020 8:55,78,ONE TIME ESP ANGELICA RIVERA GOMEZ,ROMERO JULIA,CALLE MANUEL BECERRA 14515 COL ALAMEDAS,CH-F5
3,2020,51013236,2020-02-04,1005,2/4/2020 8:15,6598,510,3.0,2/4/2020 8:02,2/4/2020 9:47,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
4,2020,51013238,2020-02-04,1026,2/4/2020 8:30,10145,510,3.5,2/4/2020 8:09,2/4/2020 9:09,60,ONE TIME CONSTRUCENTRO CHIH,MOLINA BALDERRAMA CESAR,PERIF DE LA JUVENTUD 9926 COL RESIDENCIA,CH-L5


DO NOT TOUCH THIS CODE (Fixes issue with format of `Q2 2025.csv`)

In [2]:
order_date = pd.to_datetime(remissions["order_date"], errors="coerce").dt.normalize()

def rebuild_datetime(col):
    raw = remissions[col].astype(str).str.strip()

    time_str = raw.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?\s*[APMapm]{0,2})")[0]
    time_str = time_str.str.replace(r"\.\d+", "", regex=True)

    t24 = pd.to_datetime(time_str, format="%H:%M:%S", errors="coerce")
    t24 = t24.fillna(pd.to_datetime(time_str, format="%H:%M", errors="coerce"))

    t12 = pd.to_datetime(time_str, format="%I:%M:%S %p", errors="coerce")
    t12 = t12.fillna(pd.to_datetime(time_str, format="%I:%M %p", errors="coerce"))

    t = t24.fillna(t12)
    time_only = t - t.dt.normalize()

    return order_date + time_only

remissions["typed_time"] = rebuild_datetime("typed_time")
remissions["start_time"] = rebuild_datetime("start_time")
remissions["at_plant_time"] = rebuild_datetime("at_plant_time")

In [3]:
# show rows from May 2025 
remissions_may = remissions[(remissions["order_date"].dt.month == 5) & (remissions["order_date"].dt.year == 2025)]
remissions_may.head()

,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
295158,2025,51008733,2025-05-02,1212,2025-05-02 07:00:00,7043,510,3.5,2025-05-02 06:15:15,2025-05-02 08:05:51,110,ONE TIME OCTAVIO RIOS,NORMA CHAPARRO VAZQUEZ,CALLE BOSQUE DE PIEDRA #2828 BOSQUE DE,CHG-2
295159,2025,51008735,2025-05-02,1223,2025-05-02 07:00:00,9431,510,3.5,2025-05-02 06:15:52,2025-05-02 07:55:00,100,CASA HOGAR PARA NIÑOS YIREH,CASA HOGAR PARA NIÑOS YIREH,BAHIA DE SAN QUINTIN 9513 COL RINCONADAS,CHK-3
295160,2025,51008737,2025-05-02,1227,2025-05-02 07:15:00,6605,510,3.0,2025-05-02 06:27:16,2025-05-02 08:33:00,126,LEONEL RAMIREZ JARAMILLO,LEONEL RAMIREZ JARAMILLO,CIRCUITO PODERES #10702 SAN GABRIEL ETA,CHD-4
295161,2025,51008739,2025-05-02,1423,2025-05-02 07:45:00,12193,510,6.5,2025-05-02 07:47:03,2025-05-02 09:17:00,90,JONATHAN MARQUEZ BACA,OBRAS VARIAS,CALLE ENRIQUE MULLER & LEONA VICARIO RE,CHK-5
295162,2025,51008740,2025-05-02,1214,2025-05-02 08:15:00,9431,510,2.0,2025-05-02 07:55:56,2025-05-02 08:37:00,42,INDUSTRIAS CALCITE,AV TECNOLOGICO,AV TECNOLOGICO 11708 REVOLUCION CHIHU,CHJ-6


#### Checking issue with some data corruption from plant 710 

In [4]:
# Diagnose how Q2 2025 dates are parsed around May 2-3 vs Apr 6-7 for p1 (raw q2) and p2 (remmissions)
remissions_window = remissions.loc[
    (remissions["order_date"] >= "2025-04-01")
    & (remissions["order_date"] < "2025-05-10"),
    "order_date"
].dt.date.value_counts().sort_index()
remissions_window.head(40)

raw_q2_2025 = pd.read_csv("../data/raw/BDD PRODUCCION/BDD PRODUCCION/2025/Q2 2025.csv")
raw_q2_2025["order_date_raw"] = raw_q2_2025["order_date"].astype(str).str.strip()

raw_q2_2025["p1"] = pd.to_datetime(raw_q2_2025["order_date_raw"], dayfirst=True, errors="coerce")
raw_q2_2025["p2"] = pd.to_datetime(raw_q2_2025["order_date_raw"], dayfirst=False, errors="coerce")
raw_q2_2025["num"] = pd.to_numeric(raw_q2_2025["order_date_raw"], errors="coerce")
raw_q2_2025["excel"] = pd.to_datetime(raw_q2_2025["num"], unit="D", origin="1899-12-30")

def sample_range(series, start, end):
    mask = (series >= start) & (series < end)
    cols = ["order_date_raw", "p1", "p2", "excel"]
    return raw_q2_2025.loc[mask, cols].head(20)

print("p1 May 2-3:")
display(sample_range(raw_q2_2025["p1"], "2025-05-02", "2025-05-04"))

print("p2 May 2-3:")
display(sample_range(raw_q2_2025["p2"], "2025-05-02", "2025-05-04"))

print("excel May 2-3:")
display(sample_range(raw_q2_2025["excel"], "2025-05-02", "2025-05-04"))

print("p1 Apr 6-7:")
display(sample_range(raw_q2_2025["p1"], "2025-04-06", "2025-04-08"))

print("p2 Apr 6-7:")
display(sample_range(raw_q2_2025["p2"], "2025-04-06", "2025-04-08"))

print("excel Apr 6-7:")
display(sample_range(raw_q2_2025["excel"], "2025-04-06", "2025-04-08"))

p1 May 2-3:


,order_date_raw,p1,p2,excel


p2 May 2-3:


,order_date_raw,p1,p2,excel
0,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
1,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
2,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
3,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
4,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
5,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
6,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
7,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
8,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
9,5/2/2025 0:00,2025-02-05,2025-05-02,NaT


excel May 2-3:


...

,order_date_raw,p1,p2,excel
1091,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1092,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1093,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1094,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1095,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1096,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1097,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1098,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1099,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1100,6/4/2025 0:00,2025-04-06,2025-06-04,NaT


p2 Apr 6-7:


,order_date_raw,p1,p2,excel


excel Apr 6-7:


,order_date_raw,p1,p2,excel


1. Variable Formatting

In [5]:
# Change 'order_date' and 'typed_time' to datetime format
remissions['order_date'] = pd.to_datetime(remissions['order_date'], errors='raise', format='mixed')
remissions['typed_time'] = pd.to_datetime(remissions['typed_time'], errors='raise', format='mixed')
remissions['start_time'] = pd.to_datetime(remissions['start_time'], errors='raise', format='mixed')
remissions['at_plant_time'] = pd.to_datetime(remissions['at_plant_time'], errors='raise', format='mixed')
remissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 356332 entries, 0 to 356331
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Year                 356332 non-null  int64         
 1   tkt_code             356332 non-null  int64         
 2   order_date           356332 non-null  datetime64[us]
 3   order_code           356332 non-null  int64         
 4   start_time           356332 non-null  datetime64[us]
 5   truck_code           356332 non-null  int64         
 6   ship_plant_code      356332 non-null  int64         
 7   u_Volumen            356332 non-null  float64       
 8   typed_time           356332 non-null  datetime64[us]
 9   at_plant_time        356332 non-null  datetime64[us]
 10  u_Cicle              356332 non-null  int64         
 11  name                 356331 non-null  str           
 12  Nombre del proyecto  356332 non-null  str           
 13  ship_addr_line       3562

In [6]:
# Change `ship_plant_code`, `tkt_code`, `order_code` and `truck_code` to string type
remissions['ship_plant_code'] = remissions['ship_plant_code'].astype(str)
remissions['tkt_code'] = remissions['tkt_code'].astype(str)
remissions['order_code'] = remissions['order_code'].astype(str)
remissions['truck_code'] = remissions['truck_code'].astype(str)
remissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 356332 entries, 0 to 356331
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Year                 356332 non-null  int64         
 1   tkt_code             356332 non-null  str           
 2   order_date           356332 non-null  datetime64[us]
 3   order_code           356332 non-null  str           
 4   start_time           356332 non-null  datetime64[us]
 5   truck_code           356332 non-null  str           
 6   ship_plant_code      356332 non-null  str           
 7   u_Volumen            356332 non-null  float64       
 8   typed_time           356332 non-null  datetime64[us]
 9   at_plant_time        356332 non-null  datetime64[us]
 10  u_Cicle              356332 non-null  int64         
 11  name                 356331 non-null  str           
 12  Nombre del proyecto  356332 non-null  str           
 13  ship_addr_line       3562

In [7]:
# We don't need column `Year`
remissions = remissions.drop(columns=["Year"]) # One-time use
remissions.head(1)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,51013232,2020-02-04,1073,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:00,2020-02-04 08:17:00,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1


Null/Missing data check

In [8]:
# Check for null values in each column
nulls = remissions.isnull().sum()
nulls = nulls[nulls > 0]  # keep only columns with at least 1 null

null_summary = pd.DataFrame({
    "null_count": nulls,
    "null_%": (nulls / len(remissions) * 100).round(2)
}).sort_values("null_count", ascending=False)

null_summary

,null_count,null_%
map_page,444,0.12
ship_addr_line,85,0.02
name,1,0.00


In [9]:
# It´s okay to drop rows with null values (<2%)
remissions = remissions[remissions['ship_addr_line'].notnull() & remissions['map_page'].notnull() & remissions['name'].notnull()]
print("Number of rows after dropping null values:", remissions.shape[0])

Number of rows after dropping null values: 355802


Duplicate Check

In [10]:
# Duplicate check for all time columns
typed_time_duplicates = remissions[remissions.duplicated(subset=['start_time', 'at_plant_time','truck_code'], keep=False)]
typed_time_duplicates = typed_time_duplicates.sort_values(by='typed_time', ascending=True)
typed_time_duplicates.head(10)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page


In [11]:
print("Number of duplicate rows based on start_time, at_plant_time, and truck_code:", typed_time_duplicates.shape[0])

Number of duplicate rows based on start_time, at_plant_time, and truck_code: 0


In [12]:
# It is impossible for many trucks to have the same start_time and at_plant_time
# Therefore, these are duplicates that should be removed, as they are likely to be errors in the data entry process.
remissions = remissions.drop(typed_time_duplicates.index)
print("Number of rows after dropping duplicates:", remissions.shape[0])

Number of rows after dropping duplicates: 355802


##### Outlier and Missing data Checking

In [13]:
# Check missing data over time
# Analyze hourly distribution of u_Volumen by ship_plant_code
hourly_distribution = (
    remissions
    .assign(hour_bucket=remissions["typed_time"].dt.floor("h"))
    .groupby(["hour_bucket", "ship_plant_code"], as_index=False)["u_Volumen"]
    .sum()
)

fig = px.line(
    hourly_distribution,
    x="hour_bucket",
    y="u_Volumen",
    color="ship_plant_code",
    title="Hourly Distribution of u_Volumen by Ship Plant Code (2020-2026)"
)
fig.show()

In [14]:
# Analyze hourly distribution of remission count by ship_plant_code
hourly_distribution = (
    remissions
    .assign(hour_bucket=remissions["typed_time"].dt.floor("h"))
    .groupby(["hour_bucket", "ship_plant_code"], as_index=False)
    .size()
    .rename(columns={"size": "remission_count"})
)

hourly_distribution["u_Volumen"] = hourly_distribution["remission_count"]

fig = px.line(
    hourly_distribution,
    x="hour_bucket",
    y="remission_count",
    color="ship_plant_code",
    title="Hourly Distribution of Remissions by Ship Plant Code (2020-2026)"
)
fig.show()

#### General Missing Data Observations:
- The common no-data-registered periods for all plants are:
    + `January 1st 2022` - `Feb 2nd 2022`
        + **Most likely reason**: Winter Vacations
    + `December 31 2023` - `Feb 1st 2024`
        + **Most likely reason**: Winter Vacations
    + `Mar 31 2025` - `May 2nd 2025`
    
- There are NO apparent stop from working during `2022-2023` winter vacations period nor `2024-2025`

- **Action taken:** After finishing cleaning phase I will apply imputation methods based on historic data and similar plants only for non-winter-vacations gaps (which is one). [Data_Imputation](./2.data_imputation.ipynb). As for winter gaps, probably make a winter-vacations categorical column.

#### Missing Data per plant and Outliers:
- Plant 514
    + Begins to register since `July 20 2023` 
    --- **Action Taken** ---
    + No action, TFT is not affected for this kind of "late" data

- Plant 710
    + Massive gap between `May 2023` - `Feb 2024`
    + Outlier during `June 2021`
    --- **Action Taken** ---
    + After cleaning, I will apply imputation methods based on historic data and similar plants. [Data_Imputation](./2.data_imputation.ipynb)

- Plant 512
    + Heavy downfall in remissions and volume between `April 2020` and `June 15 2020` most likely due to pandemic
    --- **Action Taken** ---
    + Because it is the only plant with this downturn, I will add a `plant-specific regime flag` for this specific plant for the model to know this event (pandemmic)
    + Outlier during `September 2024`

- Plant 510:
    + `October 2021` outlier

- Plant 511:
    + `April 2026` outlier
- Plant 717:
    + Too few data and non-operational since 2022
    --- **Action Taken** ---
    + Check percentage of data and consider deleting the whole plant

- Plant 515
    + `May 2024` and `June 2025` Outliers




#### Other Observations
- `April 2025` has no data in the given raw dataset, but there are rows for `April 6` and `April 7` of said year, also data from `May 2` and `May 3` is not appearing.
- Outliers to check:
    + Too many orders and volume in `Sept 5 2024 10:00am`
    + Some outliers for late 2025

Deleting Plant 717

In [15]:
# See percentage of plant 717 remissions count 
plant_717_count = remissions[remissions["ship_plant_code"] == "717"].shape[0]
total_count = remissions.shape[0]
print(f"Percentage of remissions from plant 717: {plant_717_count / total_count * 100:.2f}%")

Percentage of remissions from plant 717: 0.56%


In [16]:
# Get rid of plant 717 remissions since they are only 0.5% of the data and have a very different pattern than the rest of the plants, which could be due to data entry errors or a different process that is not representative of the other plants.
remissions = remissions[remissions["ship_plant_code"] != "717"]

Outlier Fixing

In [17]:
# Plant 710 remission count boxplot (row count) 
plant_710_counts = (
    remissions.loc[remissions["ship_plant_code"] == "710"]
    .groupby(remissions["typed_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)

fig = px.box(
    plant_710_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 710",
)
fig.show()


In [18]:
# Plant 710 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_710_counts["remission_count"].quantile(0.25)
Q3 = plant_710_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_710_counts[plant_710_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 710 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 710 is above the upper fence: 322


,remission_count,hour_count
0,8,169
1,9,99
2,10,35
3,11,13
4,12,4
5,13,1
6,16,1


In [19]:
# Only actual outlier looking ad the graph would be the 16 remissions of June 14th 2021 at 10am
# See all remissions from June 14th 2021 at 10am (10:00-10:59) to check for unusual repetitions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "710") &
    (remissions["typed_time"] >= "2021-06-14 10:00:00") &
    (remissions["typed_time"] < "2021-06-14 11:00:00")
]
outlier_remissions.head(16)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
75427,71089277,2021-06-14,1029,2021-06-14 08:00:00,4417,710,4.0,2021-06-14 10:05:25,2021-06-14 10:21:53,16,SAYRA LORENA HOLGUIN OCHOA,HOLGUIN OCHOA SAYRA LORENA,Pradera Lake Lote 12 Manzana A,CH-V5
75428,71089278,2021-06-14,1037,2021-06-14 10:15:00,6302,710,2.0,2021-06-14 10:09:48,2021-06-14 11:39:11,90,ONE TIME CARLOS GALVAN,HERRERA ROJAS BENJAMIN,PEDRO ROBLES 6323 COL SAUCITO,CH-N5
75429,71089279,2021-06-14,1041,2021-06-14 10:30:00,6600,710,1.5,2021-06-14 10:11:01,2021-06-14 11:08:00,57,CONSTRUCCIONES PROFESIONALES GAMA,FRAC TARRAGONA II ( CTU ),FRAC TARRAGONA II ( CTU ) S/N FRAC T,CH-N3
75430,71089281,2021-06-14,1072,2021-06-14 08:00:00,9628,710,5.5,2021-06-14 10:16:43,2021-06-14 10:23:26,7,PARCELAS CHUVISCAR,CHAVIRA FRACC AZUR FASE I,"AV. HACIENDA BONITA, CERRADA AZUR, C CIR",CH-V1
75431,71089282,2021-06-14,1072,2021-06-14 08:00:00,4411,710,5.5,2021-06-14 10:16:51,2021-06-14 10:34:40,18,PARCELAS CHUVISCAR,CHAVIRA FRACC AZUR FASE I,"AV. HACIENDA BONITA, CERRADA AZUR, C CIR",CH-V1
75432,71089283,2021-06-14,1029,2021-06-14 08:00:00,4426,710,4.0,2021-06-14 10:19:00,2021-06-14 10:34:14,15,SAYRA LORENA HOLGUIN OCHOA,HOLGUIN OCHOA SAYRA LORENA,Pradera Lake Lote 12 Manzana A,CH-V5
75433,71089284,2021-06-14,1088,2021-06-14 08:30:00,4428,710,1.5,2021-06-14 10:19:51,2021-06-14 10:35:25,16,ONE TIME BUEN FIN VICTOR MARQUEZ,JR SAMLER CONSORCIO S DE RL DE,PEDRO ZULUAGA 11400- INT 81 FRACC. VERAN,CH-V1
75434,71089285,2021-06-14,1056,2021-06-14 08:45:00,9625,710,2.0,2021-06-14 10:20:01,2021-06-14 10:36:24,16,RUBA DESARROLLOS,KDC FRACC MONTEVERDE GUARNICIONES Y,KDC FRACC MONTEVERDE S/N KDC FRACC MO,CH-U1
75435,71089286,2021-06-14,1078,2021-06-14 09:00:00,4366,710,3.0,2021-06-14 10:21:47,2021-06-14 11:01:20,40,SAUL MARQUEZ MIRAMONTES,MARQUEZ MIRAMONTES SAUL,BOSQUES DE SAN FCO,CH-O4
75436,71089288,2021-06-14,1190,2021-06-14 09:20:00,4417,710,1.5,2021-06-14 10:22:15,2021-06-14 10:39:09,17,SAYRA LORENA HOLGUIN OCHOA,SAYRA LORENA HOLGUIN OCHOA,Pradera Lake Lote 12 Manzana A,CH-V5


We'll keep plant 710's "outliers" due to legitimate data and no clues of repetition or errors

In [20]:
# Plant 510 remission count boxplot (row count) 
plant_510_counts = (
    remissions.loc[remissions["ship_plant_code"] == "510"]
    .groupby(remissions["typed_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)
fig = px.box(
    plant_510_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 510",
)
fig.show()

In [21]:
# Plant 510 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_510_counts["remission_count"].quantile(0.25)
Q3 = plant_510_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_510_counts[plant_510_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 510 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 510 is above the upper fence: 74


,remission_count,hour_count
0,11,42
1,12,13
2,13,11
3,14,6
4,15,1
5,21,1


In [22]:
# Only actual outlier looking ad the graph would be the 16 remissions of October 5th 2021 at 9am
# See all remissions from October 5th 2021 at 9am (09:00-09:59) to check for unusual repetitions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "510") &
    (remissions["typed_time"] >= "2021-10-05 09:00:00") &
    (remissions["typed_time"] < "2021-10-05 10:00:00")
]
outlier_remissions.head(21)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
79849,51041173,2021-10-05,1049,2021-10-05 07:00:00,6614,510,5.0,2021-10-05 09:00:03,2021-10-05 09:09:22,9,ONE TIME VICTOR HUGO MARQUEZ.,RAMIREZ LEONEL,PV JOSE MARTI 5508 COL GRANJAS,CH-N5
79850,51041175,2021-10-05,1055,2021-10-05 07:00:00,10145,510,1.5,2021-10-05 09:00:40,2021-10-05 09:08:24,8,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA SA D,CALLE NETZHAUALCOYOTL O SANDIVISTAS 8020,CH-M5
79851,51041176,2021-10-05,1004,2021-10-05 07:00:00,6598,510,6.5,2021-10-05 09:00:56,2021-10-05 09:07:55,7,CONSTRUCTORA INTEGRAL VALLEKAS,TIENDA HAGALO,C PASEO DE LAS FACULTADES 1405 PASEOS DE,CH-H3
79852,51041177,2021-10-05,1061,2021-10-05 07:30:00,9636,510,6.0,2021-10-05 09:01:14,2021-10-05 09:11:20,10,MATERIALES INDUSTRIALES DE,SANTA CLARA PONENTE,TABACALERAS YA RROYO EL CALORIENTO S,CH-H2
79853,51041178,2021-10-05,1004,2021-10-05 07:00:00,9631,510,6.5,2021-10-05 09:02:08,2021-10-05 09:09:39,7,CONSTRUCTORA INTEGRAL VALLEKAS,TIENDA HAGALO,C PASEO DE LAS FACULTADES 1405 PASEOS DE,CH-H3
79854,51041179,2021-10-05,1061,2021-10-05 07:30:00,10157,510,6.0,2021-10-05 09:02:26,2021-10-05 09:11:32,9,MATERIALES INDUSTRIALES DE,SANTA CLARA PONENTE,TABACALERAS YA RROYO EL CALORIENTO S,CH-H2
79855,51041181,2021-10-05,1004,2021-10-05 07:00:00,9430,510,6.0,2021-10-05 09:05:01,2021-10-05 09:11:36,6,CONSTRUCTORA INTEGRAL VALLEKAS,TIENDA HAGALO,C PASEO DE LAS FACULTADES 1405 PASEOS DE,CH-H3
79856,51041182,2021-10-05,1077,2021-10-05 08:15:00,6163,510,2.0,2021-10-05 09:05:22,2021-10-05 09:11:26,6,CONSTRUCTORA Y SUPERVISORA DE LA,BODEGA COVISA,VIALIDAD SACRAMENTO 10709 A UN COSTAD,CH-H9
79857,51041183,2021-10-05,1013,2021-10-05 08:00:00,6598,510,4.0,2021-10-05 09:08:19,2021-10-05 09:11:40,3,ALTTA HOMES NORTE,LUJAN 323/CH/21 D53005298 FRACC VIN,LUJAN 323/CH/21 D53005298 FRACC S/N,CH-K3
79858,51041184,2021-10-05,1071,2021-10-05 08:30:00,10145,510,5.0,2021-10-05 09:08:59,2021-10-05 09:11:29,3,ONE TIME ESP ANGELICA RIVERA GOMEZ,ONE TIME ESP ANGELICA RIVERA GOMEZ,CD OJINAGA 129 COL REVOLUCION,CH-J3


CURR_FLAG

In [23]:
# CURRENT TASK: Preguntar mañana sobre tiempo de recarga de camiones para saber si quitar ciertas columnas 
# Potentially delete some columns with apparently impossible recharge times for trucks causing outliers


In [24]:
# Plant 511 remission count boxplot (row count) 
plant_511_counts = (
    remissions.loc[remissions["ship_plant_code"] == "511"]
    .groupby(remissions["typed_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)
fig = px.box(
    plant_511_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 511",
)
fig.show()

In [25]:
# Plant 511 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_511_counts["remission_count"].quantile(0.25)
Q3 = plant_511_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_511_counts[plant_511_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 511 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 511 is above the upper fence: 48


,remission_count,hour_count
0,10,39
1,11,5
2,12,2
3,13,1
4,22,1


In [26]:
# Only actual outlier looking ad the graph would be the 16 remissions of April 7th 2026 at 4pm
# See all remissions from April 7th 2026 at 4pm (16:00-16:59) to check for unusual repetitions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "511") &
    (remissions["typed_time"] >= "2026-04-07 16:00:00") &
    (remissions["typed_time"] < "2026-04-07 17:00:00")
]
outlier_remissions.head(22)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
345268,51167685,2026-04-07,1113,2026-04-07 16:45:00,13040,511,1.0,2026-04-07 16:01:33,2026-04-07 16:12:25,11,MARIO CONTRERAS LARA,FRAC VILANOVA IV (CTU),FRAC VILANOVA IV (CTU) S/N FRAC VILAN,CHN-17
345269,51167687,2026-04-07,1385,2026-04-07 22:00:00,4426,511,6.0,2026-04-07 16:11:04,2026-04-07 16:14:48,3,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16
345270,51167688,2026-04-07,1385,2026-04-07 22:00:00,13050,511,6.0,2026-04-07 16:11:24,2026-04-07 16:15:04,4,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16
345271,51167689,2026-04-07,1385,2026-04-07 22:00:00,13042,511,6.0,2026-04-07 16:11:39,2026-04-07 16:15:08,4,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16
345272,51167690,2026-04-07,1385,2026-04-07 22:00:00,9636,511,6.0,2026-04-07 16:11:55,2026-04-07 16:15:10,4,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16
345273,51167691,2026-04-07,1385,2026-04-07 22:00:00,9432,511,6.0,2026-04-07 16:12:16,2026-04-07 16:15:12,3,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16
345274,51167692,2026-04-07,1385,2026-04-07 22:00:00,13040,511,6.0,2026-04-07 16:12:37,2026-04-07 16:15:14,3,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16
345275,51167693,2026-04-07,1385,2026-04-07 22:00:00,13010,511,6.0,2026-04-07 16:12:52,2026-04-07 16:15:17,3,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16
345276,51167694,2026-04-07,1385,2026-04-07 22:00:00,6600,511,6.0,2026-04-07 16:13:12,2026-04-07 16:17:19,4,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16
345277,51167695,2026-04-07,1385,2026-04-07 22:00:00,12832,511,6.0,2026-04-07 16:14:00,2026-04-07 16:17:49,3,ARQUITECTURA HABITACIONAL E,1672 AIIG INVENTARIO III,BLVD JUAN PABLO II 13901 TABALOAPA CH,CHK-16


In [27]:
# Impute constant-volume anomaly for plant 511 on 2026-04-07 16:00-16:59 by sampling historical per-hour volumes
mask_511 = (remissions['ship_plant_code'] == '511') & \
           (remissions['typed_time'] >= '2026-04-07 16:00:00') & \
           (remissions['typed_time'] <  '2026-04-07 17:00:00')
print('Rows matching 511 anomaly:', mask_511.sum())
if mask_511.sum() > 0:
    idxs = remissions.loc[mask_511].index
    p_hist = remissions[(remissions['ship_plant_code'] == '511') & (~mask_511)].copy()
    target_hour = 16
    target_dow = pd.to_datetime('2026-04-07').weekday()
    pool = p_hist[p_hist['typed_time'].dt.hour.eq(target_hour) & p_hist['typed_time'].dt.weekday.eq(target_dow)]['u_Volumen']
    if pool.empty:
        pool = p_hist[p_hist['typed_time'].dt.hour.eq(target_hour)]['u_Volumen']
    if pool.empty:
        pool = p_hist['u_Volumen'].tail(500)
    if pool.empty:
        pool = remissions[remissions['typed_time'].dt.hour.eq(target_hour)]['u_Volumen'].dropna()
    if pool.empty:
        print('No historical pool found; skipping imputation for 511')
    else:
        sampled = pool.sample(n=len(idxs), replace=True, random_state=123).values
        remissions.loc[idxs, 'u_Volumen_before_impute'] = remissions.loc[idxs, 'u_Volumen']
        remissions.loc[idxs, 'u_Volumen'] = sampled
        print(f'Imputed {len(idxs)} rows for plant 511 by sampling (pool size={len(pool)})')
else:
    print('No rows to impute for plant 511')

Rows matching 511 anomaly: 22
Imputed 22 rows for plant 511 by sampling (pool size=443)


In [28]:
# Plant 515 remission count boxplot (row count) 
plant_515_counts = (
    remissions.loc[remissions["ship_plant_code"] == "515"]
    .groupby(remissions["typed_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)

fig = px.box(
    plant_515_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 515",
)
fig.show()


In [29]:
# Plant 515 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_515_counts["remission_count"].quantile(0.25)
Q3 = plant_515_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_515_counts[plant_515_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 515 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 515 is above the upper fence: 264


,remission_count,hour_count
0,8,183
1,9,60
2,10,15
3,11,2
4,12,1
5,13,2
6,19,1


In [30]:
# Only actual outlier looking ad the graph would be the 16 remissions of June 14th 2025 at 1pm
# See all remissions from June 14th 2025 at 1pm (13:00-13:59) to check for unusual repetitions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "515") &
    (remissions["typed_time"] >= "2025-06-14 13:00:00") &
    (remissions["typed_time"] < "2025-06-14 14:00:00")
]
outlier_remissions.head(21)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page,u_Volumen_before_impute
307399,51583492,2025-06-14,1218,2025-06-14 08:15:00,6163,515,1.0,2025-06-14 13:03:00,2025-06-14 13:04:00,1,JOSE ABRAHAM RODRIGUEZ MORALES,RUBA MORTERO BARDAS FRACC SAN,FRACC SAN AGUSTIN BARDAS SN FRACC SAN,CHN-21,NaN
307400,51583493,2025-06-14,1235,2025-06-14 09:15:00,6163,515,1.0,2025-06-14 13:05:00,2025-06-14 13:16:00,11,COMERCIALIZADORA PASO DEL NORTE DE,FRAC PASEO DE LAS FLORES,PASEO GRANO DE ORO Y PASEO DEL SUIZO S/,CHQ-17,NaN
307401,51583494,2025-06-14,1221,2025-06-14 08:15:00,12829,515,1.0,2025-06-14 13:05:00,2025-06-14 13:17:00,12,RUBA DESARROLLOS,LUIS VELEZ JARDINES DE SAN AGUSTUN,LUIS VELEZ JARDINES DE SAN AGUSTIN 2 SN,CHR-21,NaN
307402,51583495,2025-06-14,1167,2025-06-14 09:15:00,12832,515,1.5,2025-06-14 13:09:00,2025-06-14 13:12:00,3,RUBA DESARROLLOS,CONSTRUCTER SAN AGUSTIN II BARDAS D,CONSTRUCTER SAN AGUSTIN II BARDAS DUPLEX,CHR-21,NaN
307403,51583500,2025-06-14,1256,2025-06-14 12:45:00,10154,515,3.0,2025-06-14 13:14:00,2025-06-14 13:19:00,5,EDGAR DURAN VALENZUELA,EDGAR DURAN VALENZUELA,PUERTA DE CHIHUAHUA,CHW-20,NaN
307404,51583501,2025-06-14,1168,2025-06-14 09:30:00,13219,515,1.0,2025-06-14 13:14:00,2025-06-14 13:17:00,3,RUBA DESARROLLOS,FABRINFRA JARDINES DE SAN AGUSTIN 2,FABRINFRA JARDINES DE SAN AGUSTIN 2 47 V,CHR-21,NaN
307405,51583502,2025-06-14,1220,2025-06-14 09:45:00,6600,515,3.0,2025-06-14 13:16:00,2025-06-14 13:23:00,7,RUBA DESARROLLOS,JOSE RODRIGUEZ SAN AGUSTIN 2 E2,SAN AGUSTIN 2 E2 BARDAS PERIM SN SAN,CHN-21,NaN
307406,51583503,2025-06-14,1173,2025-06-14 10:00:00,6163,515,3.5,2025-06-14 13:16:00,2025-06-14 13:24:00,8,RUBA DESARROLLOS,FABRINFRA JARDINES DE SAN AGUSTIN 2,FABRINFRA JARDINES DE SAN AGUSTIN 2 47 V,CHR-21,NaN
307407,51583504,2025-06-14,1222,2025-06-14 10:15:00,12829,515,1.0,2025-06-14 13:17:00,2025-06-14 13:23:00,6,RUBA DESARROLLOS,LIZBETH SALGADO SAN AGUSTIN 2 E2 BA,LIZBETH SALGADO SAN AGUSTIN 2 E2 BARDAS,CHR-21,NaN
307408,51583505,2025-06-14,1119,2025-06-14 10:30:00,13219,515,4.5,2025-06-14 13:17:00,2025-06-14 13:27:00,10,ONE TIME PROMOCIONES HUGO T,JORGE HERNANDEZ,VALLE DORADO,CHN-21,NaN


In [31]:
# Plant 512 remission count boxplot (row count) 
plant_512_counts = (
    remissions.loc[remissions["ship_plant_code"] == "512"]
    .groupby(remissions["typed_time"].dt.floor("h"))
    .size()
    .reset_index(name="remission_count")
)

fig = px.box(
    plant_512_counts,
    y="remission_count",
    title="Boxplot of Hourly Remission Count for Plant 512",
)
fig.show()


In [32]:
# Plant 512 amount of times that remission count has been above the upper fence (Q3 + 1.5*IQR)
Q1 = plant_512_counts["remission_count"].quantile(0.25)
Q3 = plant_512_counts["remission_count"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = plant_512_counts[plant_512_counts["remission_count"] > upper_fence]
print(f"Number of hours where remission count for plant 512 is above the upper fence: {outliers.shape[0]}")

outlier_table = (
    outliers["remission_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
outlier_table.columns = ["remission_count", "hour_count"]

display(outlier_table)

Number of hours where remission count for plant 512 is above the upper fence: 20


,remission_count,hour_count
0,14,15
1,15,3
2,16,1
3,31,1


In [33]:
# Only actual outlier looking at the graph would be the 31 remissions of September 5th 2024 at 10am
# See all remissions from September 5th 2024 at 10am (10:00-10:59) to check for unusual repetitions
outlier_remissions = remissions[
    (remissions["ship_plant_code"] == "512") &
    (remissions["typed_time"] >= "2024-09-05 10:00:00") &
    (remissions["typed_time"] < "2024-09-05 11:00:00")
]
outlier_remissions.head(22)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page,u_Volumen_before_impute
257155,51253103,2024-09-05,1007,2024-09-05 05:00:00,12863,512,6.0,2024-09-05 10:17:37,2024-09-05 10:21:06,4,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257156,51253104,2024-09-05,1007,2024-09-05 05:00:00,10154,512,6.0,2024-09-05 10:19:51,2024-09-05 10:21:38,2,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257157,51253105,2024-09-05,1007,2024-09-05 05:00:00,7044,512,6.0,2024-09-05 10:20:06,2024-09-05 10:22:03,2,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257158,51253106,2024-09-05,1007,2024-09-05 05:00:00,7039,512,6.0,2024-09-05 10:20:20,2024-09-05 10:24:40,4,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257159,51253107,2024-09-05,1007,2024-09-05 05:00:00,6302,512,6.0,2024-09-05 10:20:58,2024-09-05 11:20:33,60,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257160,51253108,2024-09-05,1007,2024-09-05 05:00:00,12863,512,6.0,2024-09-05 10:21:24,2024-09-05 10:26:13,5,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257161,51253109,2024-09-05,1007,2024-09-05 05:00:00,10154,512,6.0,2024-09-05 10:21:54,2024-09-05 10:26:26,5,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257162,51253110,2024-09-05,1007,2024-09-05 05:00:00,7044,512,6.0,2024-09-05 10:22:15,2024-09-05 10:26:38,4,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257163,51253111,2024-09-05,1007,2024-09-05 05:00:00,9428,512,6.0,2024-09-05 10:23:15,2024-09-05 10:27:08,4,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN
257164,51253112,2024-09-05,1007,2024-09-05 05:00:00,9429,512,6.0,2024-09-05 10:23:35,2024-09-05 10:27:46,4,ARQUITECTURA HABITACIONAL E,1130 SUPRA CHIHUAHUA,AV TECNOLOGICO Y GUILLERMO PRIETO LUJAN,CHF-2,NaN


Impute new volume values for this repetitive 6m3 volume per row for the Sept 5th 2025 10am

In [34]:
# Impute constant-volume anomaly for plant 512 on 2024-09-05 10:00-10:59 by sampling historical per-hour volumes
mask = (remissions['ship_plant_code'] == '512') & \
       (remissions['typed_time'] >= '2024-09-05 10:00:00') & \
       (remissions['typed_time'] <  '2024-09-05 11:00:00')
print('Rows matching anomaly:', mask.sum())
if mask.sum() > 0:
    idxs = remissions.loc[mask].index
    # historical pool for plant 512 excluding anomalous hour
    p_hist = remissions[(remissions['ship_plant_code'] == '512') & (~mask)].copy()
    # prefer same hour-of-day and same weekday
    target_hour = 10
    target_dow = pd.to_datetime('2024-09-05').weekday()
    pool = p_hist[p_hist['typed_time'].dt.hour.eq(target_hour) & p_hist['typed_time'].dt.weekday.eq(target_dow)]['u_Volumen']
    # fallbacks
    if pool.empty:
        pool = p_hist[p_hist['typed_time'].dt.hour.eq(target_hour)]['u_Volumen']
    if pool.empty:
        pool = p_hist['u_Volumen'].tail(500)
    if pool.empty:
        pool = remissions[remissions['typed_time'].dt.hour.eq(target_hour)]['u_Volumen'].dropna()
    if pool.empty:
        print('No historical pool found; skipping imputation')
    else:
        # sample with replacement so each row gets a variable historical-like value
        sampled = pool.sample(n=len(idxs), replace=True, random_state=42).values
        remissions.loc[idxs, 'u_Volumen_before_impute'] = remissions.loc[idxs, 'u_Volumen']
        remissions.loc[idxs, 'u_Volumen'] = sampled
        print(f'Imputed {len(idxs)} rows by sampling from historical pool (size={len(pool)})')
else:
    print('No rows to impute')

Rows matching anomaly: 31
Imputed 31 rows by sampling from historical pool (size=1938)


In [35]:
# Drop the temporary 'u_Volumen_before_impute' column if it exists
if 'u_Volumen_before_impute' in remissions.columns:
    remissions = remissions.drop(columns=['u_Volumen_before_impute'])

    

Inconsistencies Checking

In [36]:
remissions.describe()

,order_date,start_time,u_Volumen,typed_time,at_plant_time,u_Cicle
count,353797,353797,353797.000000,353797,353797,353797.000000
mean,2023-05-01 14:18:41.015724,2023-05-02 01:54:03.998694,3.186254,2023-05-02 01:42:27.107600,2023-05-02 02:59:15.589623,77.137443
min,2020-02-04 00:00:00,2020-02-04 05:00:00,0.340000,2020-02-04 05:30:00,2020-02-04 06:44:00,1.000000
25%,2021-10-21 00:00:00,2021-10-21 08:00:00,1.500000,2021-10-21 07:18:07,2021-10-21 08:56:44,56.000000
50%,2023-05-18 00:00:00,2023-05-18 16:00:00,3.000000,2023-05-18 15:41:03,2023-05-18 16:56:47,74.000000
75%,2024-11-04 00:00:00,2024-11-04 15:30:00,5.000000,2024-11-04 14:06:02,2024-11-04 15:19:09,96.000000
max,2026-04-24 00:00:00,2026-04-24 11:30:00,6.500000,2026-04-24 10:47:56,2026-04-24 10:49:00,209.000000
std,NaN,NaN,1.784677,NaN,NaN,35.836594


In [37]:
# Graph the distribution of u_Cicle to see if there are orders that were not completed or any unreal value
fig = px.histogram(remissions, x='u_Cicle', nbins=len(remissions['u_Cicle'].unique()), title='Distribution of u_Cicle')
fig.show()

In [38]:
# See amount of remissions per plant with u_Cicle equal to 1 (include percentage)
cicle_1_counts = remissions[remissions['u_Cicle'] == 1]['ship_plant_code'].value_counts().reset_index()
cicle_1_counts.columns = ['ship_plant_code', 'count']
total_cicle1 = cicle_1_counts['count'].sum()
cicle_1_counts['percentage'] = (cicle_1_counts['count'] / total_cicle1 * 100).round(2)
display(cicle_1_counts)

,ship_plant_code,count,percentage
0,512,4220,42.75
1,515,1715,17.37
2,514,1458,14.77
3,710,1372,13.90
4,511,783,7.93
5,510,324,3.28


In [39]:
# Boxplot of u_Cicle to check for outliers
fig = px.box(remissions, y='u_Cicle', title='Boxplot of u_Cicle')
fig.show()

#### Volume distribution check for outliers

In [40]:
# Calculate counts and percentages for u_Volumen
counts = remissions['u_Volumen'].value_counts().sort_index()
percentages = (counts / counts.sum()) * 100

# Histogram of frequency of each unique volume value to check for outliers
x_labels = counts.index.astype(str)  # treat volumes as categorical so bars are wider
fig = px.bar(x=x_labels, y=counts.values, title='Distribution of Volume per Truck',
             labels={'x':'u_Volumen','y':'count'}, text=counts.values)
fig.update_traces(
    textposition='outside',
    texttemplate='%{y}',
    customdata=percentages.values,
    hovertemplate='<b>Volume:</b> %{x}<br><b>Count:</b> %{y}<br><b>Percentage:</b> %{customdata:.2f}%<extra></extra>'
)
fig.update_layout(
    bargap=0.1,        # reduce gap between bars
    xaxis_tickangle=45,
    width=1000
)
fig.show()

In [41]:
# print unique values per column
remissions.nunique()

tkt_code               333871
order_date               1755
order_code                794
start_time              70343
truck_code                150
ship_plant_code             6
u_Volumen                  34
typed_time             347149
at_plant_time          344907
u_Cicle                   209
name                     2060
Nombre del proyecto     23562
ship_addr_line          73357
map_page                  953
dtype: int64

In [42]:
# Print order_code values that are duplicated to check for potential data entry errors
order_code_counts = remissions['order_code'].value_counts()
duplicate_order_codes = order_code_counts[order_code_counts > 1]
print("Duplicate order_code values and their counts:", duplicate_order_codes)

# print sum of duplicated order_code values to see how many rows are affected by potential data entry errors
total_duplicates = duplicate_order_codes.sum()
print(f"Total rows affected by duplicate order_code values: {total_duplicates}")

Duplicate order_code values and their counts: order_code
1113    1295
1086    1219
1070    1195
1107    1194
1147    1184
        ... 
1509       2
1515       2
1490       2
1492       2
1498       2
Name: count, Length: 754, dtype: int64
Total rows affected by duplicate order_code values: 353757


In [43]:
# print oldest typed_time row with order code 1113 and newest
order_1113 = remissions[remissions['order_code'] == '1113']
print("Oldest typed_time for order_code 1113:", order_1113['typed_time'].min())
print("Newest typed_time for order_code 1113:", order_1113['typed_time'].max())

Oldest typed_time for order_code 1113: 2020-02-05 08:25:00
Newest typed_time for order_code 1113: 2026-04-23 15:04:28


Order_code parece carecer de significado, se puede eliminar la columna sin problemas

In [44]:
# Drop order_code column since it means nothing relevant
remissions = remissions.drop(columns=['order_code'])
remissions.head(1)

,tkt_code,order_date,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,51013232,2020-02-04,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:00,2020-02-04 08:17:00,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1


In [45]:
#Looks like the tkt_codes can be repeated after years of orders.
remissions[remissions['tkt_code'].duplicated(keep=False)].sort_values('tkt_code', ascending=True).head(6)


,tkt_code,order_date,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
1,51013233,2020-02-04,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:00,2020-02-04 08:26:00,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
310446,51013233,2025-08-02,2025-08-02 07:00:00,9430,510,2.0,2025-08-02 06:46:30,2025-08-02 07:46:17,60,ONE TIME PROMOCIONES OCTAVIO RIOS,MARGARITO ROMERO,ING. CARRILLO 16344 COL TRAHUMARA,CHJ-3
3,51013236,2020-02-04,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:00,2020-02-04 09:47:00,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
310448,51013236,2025-08-02,2025-08-02 07:00:00,7043,510,4.0,2025-08-02 06:54:06,2025-08-02 08:14:23,80,JONATHAN MARQUEZ BACA,OBRAS VARIAS,ARROYO NARAGUA #2225 LOS ARROYOS,CHD-5
5,51013239,2020-02-04,2020-02-04 09:00:00,6611,510,6.0,2020-02-04 08:36:00,2020-02-04 09:30:00,54,JULIO ARMANDO HINOJOS ENRIQUEZ,FRAC CALZADA DEL BOSQUE AGH,FRAC CALZADA DEL BOSQUE AGH FRAC CAL,CH-H3
310450,51013239,2025-08-02,2025-08-02 08:00:00,9430,510,6.5,2025-08-02 07:46:56,2025-08-02 09:03:00,77,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,MONTE HIMALAYA 4341 QUINTAS CAROLINA,CHH-8


We will now check the consistency of remissions per plant

In [46]:
# Lets see order ratios between plants
counts = remissions['ship_plant_code'].value_counts()
percentages = (counts / counts.sum()) * 100
print(counts, '\t', percentages.round(2)) # percentages
print("total:", counts.sum())

ship_plant_code
512    82691
510    71417
511    61025
515    52454
710    45916
514    40294
Name: count, dtype: int64 	 ship_plant_code
512    23.37
510    20.19
511    17.25
515    14.83
710    12.98
514    11.39
Name: count, dtype: float64
total: 353797


In [47]:
# See lowest and highest datetime for each plant
for plant_code in remissions['ship_plant_code'].unique():
    plant = remissions[remissions['ship_plant_code'] == plant_code]
    print(f"Plant {plant_code}:")
    print(f"  Lowest datetime: {plant['start_time'].min()}")
    print(f"  Highest datetime: {plant['start_time'].max()}")

Plant 510:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 09:30:00
Plant 511:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 10:00:00
Plant 512:
  Lowest datetime: 2020-02-04 05:00:00
  Highest datetime: 2026-04-24 11:30:00
Plant 515:
  Lowest datetime: 2020-02-04 08:30:00
  Highest datetime: 2026-04-24 10:00:00
Plant 710:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 10:15:00
Plant 514:
  Lowest datetime: 2022-07-20 07:00:00
  Highest datetime: 2026-04-24 10:45:00


#### Exporting Cleaned Dataset

In [48]:
# Has to be xlsx because of datetime format
remissions.to_excel("../data/processed/remissions_db_cleaned.xlsx", index=False)